# PMOF action timelines per person

This notebook plots the temporal action sequence of every annotated person in every PMOF recording. Each bar represents one `(recording, track ID)` pair; stacked segments run from the first observed frame at the bottom to the last observed frame at the top.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Patch

# Set these paths for this workstation before running the notebook.
PMOF_CODE_DIR = Path(r"C:\Users\stell\Desktop\pmof-code")
DATA_BASE_DIR = Path(r"C:\path\to\PMOF")

if not PMOF_CODE_DIR.is_dir():
    raise FileNotFoundError(f"PMOF code directory not found: {PMOF_CODE_DIR}")
if not (DATA_BASE_DIR / "images").is_dir() or not (DATA_BASE_DIR / "annotations").is_dir():
    raise FileNotFoundError(
        "DATA_BASE_DIR must contain both 'images' and 'annotations' directories. "
        f"Current value: {DATA_BASE_DIR}"
    )

sys.path.insert(0, str(PMOF_CODE_DIR))

from src.data import list_record_ids, read_annotation, recid_to_annpath, recordid_to_imageids
from src.visualization import VIZ_PARAMS

ACTION_COLORS = VIZ_PARAMS["gt_bbox_colors"]
ACTION_ORDER = ["seated", "seated_ground", "standing", "lying"]

In [ ]:
def frame_number(image_id: str) -> int:
    """Extract the numeric frame index from an ID such as ``rec12_000345``."""
    return int(image_id.rsplit("_", maxsplit=1)[1])


def load_person_actions(data_base_dir: Path) -> pd.DataFrame:
    """Read every annotated person action with PMOF's dataset and annotation helpers."""
    action_rows = []

    for record_id in list_record_ids(data_base_dir):
        annotation_path = recid_to_annpath(record_id, data_base_dir)
        for image_id in recordid_to_imageids(record_id, data_base_dir):
            for annotation in read_annotation(annotation_path, image_id):
                if annotation.category_name != "person" or annotation.action is None:
                    continue
                action_rows.append(
                    {
                        "record_id": record_id,
                        "frame": frame_number(image_id),
                        "track_id": annotation.track_id,
                        "action": annotation.action,
                    }
                )

    frame_actions = pd.DataFrame(
        action_rows, columns=["record_id", "frame", "track_id", "action"]
    )
    if frame_actions.empty:
        raise ValueError("No person annotations with an action attribute were found.")

    unsupported_actions = sorted(set(frame_actions["action"]) - set(ACTION_ORDER))
    if unsupported_actions:
        raise ValueError(
            "Expected only PMOF actions "
            f"{ACTION_ORDER}; found: {', '.join(unsupported_actions)}"
        )
    if frame_actions.duplicated(["record_id", "frame", "track_id"]).any():
        raise ValueError("A person has multiple action annotations in the same frame.")

    return frame_actions.sort_values(["record_id", "track_id", "frame"]).reset_index(drop=True)


frame_actions = load_person_actions(DATA_BASE_DIR)
frame_actions.head()

In [ ]:
def consecutive_action_runs(person_frames: pd.DataFrame) -> list[dict]:
    """Collapse consecutive equal actions while retaining chronological order."""
    runs = []
    current_action = None
    run_start = None
    run_end = None
    frame_count = 0

    for row in person_frames.sort_values("frame").itertuples(index=False):
        if row.action != current_action:
            if current_action is not None:
                runs.append(
                    {
                        "action": current_action,
                        "start_frame": run_start,
                        "end_frame": run_end,
                        "frame_count": frame_count,
                    }
                )
            current_action = row.action
            run_start = row.frame
            frame_count = 0
        run_end = row.frame
        frame_count += 1

    if current_action is not None:
        runs.append(
            {
                "action": current_action,
                "start_frame": run_start,
                "end_frame": run_end,
                "frame_count": frame_count,
            }
        )
    return runs


run_rows = []
for (record_id, track_id), person_frames in frame_actions.groupby(["record_id", "track_id"], sort=True):
    for run in consecutive_action_runs(person_frames):
        run_rows.append({"record_id": record_id, "track_id": track_id, **run})

action_runs = pd.DataFrame(run_rows)
action_runs.head(10)

In [ ]:
record_order = list_record_ids(DATA_BASE_DIR)
track_order = [
    (record_id, track_id)
    for record_id in record_order
    for track_id in sorted(
        frame_actions.loc[frame_actions["record_id"] == record_id, "track_id"].unique()
    )
]

fig, ax = plt.subplots(figsize=(max(12, 0.35 * len(track_order)), 6), dpi=150)

for position, (record_id, track_id) in enumerate(track_order):
    person_runs = action_runs.loc[
        (action_runs["record_id"] == record_id)
        & (action_runs["track_id"] == track_id)
    ].sort_values("start_frame")
    bottom = 0

    for run in person_runs.itertuples(index=False):
        ax.bar(
            position,
            run.frame_count,
            bottom=bottom,
            width=0.85,
            color=ACTION_COLORS[run.action],
            edgecolor="none",
            antialiased=False,
        )
        bottom += run.frame_count

for position in range(1, len(track_order)):
    if track_order[position][0] != track_order[position - 1][0]:
        ax.axvline(position - 0.5, color="black", linewidth=0.6, alpha=0.6)

observed_actions = set(action_runs["action"])
legend_handles = [
    Patch(facecolor=ACTION_COLORS[action], label=action.replace("_", " "))
    for action in ACTION_ORDER
    if action in observed_actions
]
ax.legend(handles=legend_handles, title="Action", ncols=min(4, len(legend_handles)))
ax.set_xticks(range(len(track_order)))
ax.set_xticklabels([f"{record_id}\nP{track_id}" for record_id, track_id in track_order], rotation=90)
ax.set_xlabel("Recording and person track ID")
ax.set_ylabel("Annotated frames per person (chronological from bottom to top)")
ax.set_title("PMOF action timelines by person")
ax.set_xlim(-0.6, len(track_order) - 0.4)
ax.grid(axis="y", linewidth=0.4, alpha=0.35)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()